# Task 089: pyTFM — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Traction force microscopy using FTTC (Fourier Transform Traction Cytometry)

**Paper**: empymod: Open-source full 3D electromagnetic modeller for 1D VTI media
**Repository**: https://github.com/emsig/empymod

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 22.92 dB
- **SSIM**: 0.6963
- **Evaluation**: 2D traction force field reconstruction — 从凝胶基底变形场恢复细胞牵引力 (CC=0.972)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
pyTFM - Traction Force Microscopy Inverse Problem
===================================================
Task: Recover cell traction forces from gel surface displacement fields
      using Fourier Transform Traction Cytometry (FTTC).

Inverse Problem:
    Given measured displacement field (u, v) on an elastic substrate surface,
    recover the traction force field (tx, ty) that caused those displacements.

Forward Model (Boussinesq):
    (u, v) = G * (tx, ty)   [convolution via Fourier-space Green's function]
    where G is the Boussinesq elastic Green's function for a half-space.

Inverse Solver:
    (tx, ty) = G^{-1} * (u, v)  [Fourier-space inversion with spatial filtering]

Repo: https://github.com/fabrylab/pyTFM
Paper: Butler et al. (2002), Traction fields, moments, and strain energy that
       cells exert on their surroundings. Am J Physiol Cell Physiol.

Usage:
    /data/yjh/pyTFM_env/bin/python pyTFM_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_correlation_coefficient`

In [ ]:
def compute_correlation_coefficient(ref, test):
    """Compute Pearson correlation coefficient."""
    ref_flat = ref.ravel()
    test_flat = test.ravel()
    if np.std(ref_flat) == 0 or np.std(test_flat) == 0:
        return 0.0
    return float(np.corrcoef(ref_flat, test_flat)[0, 1])

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`, `generate_synthetic_traction_field`, `load_or_generate_data`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. Forward Operator (Traction → Displacement)
# ═══════════════════════════════════════════════════════════
def forward_operator(tx, ty, pixelsize, young, sigma=0.49):
    """
    Boussinesq forward model (infinite half-space): compute surface displacements
    from traction forces using Fourier-space Green's function.

    CRITICAL: The padding and wave vector conventions MUST exactly match
    ffttc_traction in pyTFM/TFM_functions.py for perfect round-trip consistency.

    The inverse solver computes:  T = K_inv * FFT(u * pixelsize1)
    So the forward is:  u = IFFT(K * T_ft) / pixelsize1

    where K_inv is the elastic stiffness kernel and K = K_inv^{-1} is the
    compliance (Green's function) kernel.

    Parameters:
        tx, ty: 2D arrays, traction fields in Pa
        pixelsize: float, pixel size in meters (same for pixelsize1 and pixelsize2)
        young: float, Young's modulus in Pa
        sigma: float, Poisson's ratio

    Returns:
        u, v: 2D arrays, displacement fields in PIXELS
    """
    ax1_length, ax2_length = tx.shape
    max_ind = int(max(ax1_length, ax2_length))
    if max_ind % 2 != 0:
        max_ind += 1

    # MUST match inverse solver's padding convention: top-left placement
    tx_expand = np.zeros((max_ind, max_ind))
    ty_expand = np.zeros((max_ind, max_ind))
    tx_expand[:ax1_length, :ax2_length] = tx
    ty_expand[:ax1_length, :ax2_length] = ty

    # Wave vectors — MUST match ffttc_traction exactly
    # In ffttc_traction: kx = ... * 2*pi, then k = sqrt(kx^2 + ky^2) / (pixelsize2 * max_ind)
    kx1 = np.array([list(range(0, int(max_ind / 2), 1)), ] * int(max_ind))
    kx2 = np.array([list(range(-int(max_ind / 2), 0, 1)), ] * int(max_ind))
    kx = np.append(kx1, kx2, axis=1) * 2 * np.pi
    ky = np.transpose(kx)
    k = np.sqrt(kx ** 2 + ky ** 2) / (pixelsize * max_ind)

    # Angle (same as inverse solver)
    alpha = np.arctan2(ky, kx)
    alpha[0, 0] = np.pi / 2

    # The inverse solver's K_inv kernel:
    # kix = (k*E)/(2*(1-sigma^2)) * (1-sigma + sigma*cos(alpha)^2)
    # kiy = (k*E)/(2*(1-sigma^2)) * (1-sigma + sigma*sin(alpha)^2)
    # kid = (k*E)/(2*(1-sigma^2)) * sigma*sin(alpha)*cos(alpha)
    #
    # Forward kernel G = K_inv^{-1}: we need to invert the 2x2 system at each k:
    # [kix kid] [u_ft]   [tx_ft]
    # [kid kiy] [v_ft] = [ty_ft]  (after scaling u by pixelsize1)
    #
    # So: [u_ft] = 1/det * [ kiy -kid] [tx_ft]
    #     [v_ft]            [-kid  kix] [ty_ft]
    # where det = kix*kiy - kid^2

    kix = ((k * young) / (2 * (1 - sigma ** 2))) * (1 - sigma + sigma * np.cos(alpha) ** 2)
    kiy = ((k * young) / (2 * (1 - sigma ** 2))) * (1 - sigma + sigma * np.sin(alpha) ** 2)
    kid = ((k * young) / (2 * (1 - sigma ** 2))) * (sigma * np.sin(alpha) * np.cos(alpha))

    # Zero out cross terms at Nyquist (same as inverse solver)
    kid[:, int(max_ind / 2)] = np.zeros(max_ind)
    kid[int(max_ind / 2), :] = np.zeros(max_ind)

    # Determinant of the K_inv 2x2 matrix
    det = kix * kiy - kid ** 2

    # Avoid division by zero at DC
    det[0, 0] = 1.0  # will be zeroed out anyway

    # Green's function (inverse of K_inv)
    g11 = kiy / det
    g12 = -kid / det
    g22 = kix / det

    # FFT of traction fields
    tx_ft = scipy.fft.fft2(tx_expand)
    ty_ft = scipy.fft.fft2(ty_expand)

    # Displacement in Fourier space (in meters, since inverse uses u*pixelsize1)
    u_ft = g11 * tx_ft + g12 * ty_ft
    v_ft = g12 * tx_ft + g22 * ty_ft

    # Zero DC component (mean displacement is unconstrained)
    u_ft[0, 0] = 0
    v_ft[0, 0] = 0

    # Back to real space
    u = scipy.fft.ifft2(u_ft).real
    v = scipy.fft.ifft2(v_ft).real

    # Cut to original size (MUST match inverse solver's placement)
    u_cut = u[:ax1_length, :ax2_length]
    v_cut = v[:ax1_length, :ax2_length]

    # The inverse solver multiplies u by pixelsize1 before FFT:
    #   u_ft_inv = FFT(u_pixels * pixelsize1)
    #   tx_ft = kix * u_ft_inv + kid * v_ft_inv
    #
    # So the forward result in meters is: u_meters = u_cut (from above)
    # And u_pixels = u_meters / pixelsize1
    return u_cut / pixelsize, v_cut / pixelsize

# ═══════════════════════════════════════════════════════════
# 3. Data Generation (Synthetic Benchmark)
# ═══════════════════════════════════════════════════════════
def generate_synthetic_traction_field(N=64):
    """
    Generate a synthetic cell-like traction force dipole pattern.
    Models a single cell contracting on the substrate — two opposing
    Gaussian-shaped force patches (contractile dipole).

    Returns:
        tx_gt, ty_gt: ground truth traction fields in Pa
    """
    y, x = np.mgrid[:N, :N]
    cx, cy = N // 2, N // 2

    # Two Gaussian blobs offset from center (contractile dipole)
    sigma_blob = N / 8.0
    offset = N / 5.0

    # Left blob (pulling right)
    g_left = np.exp(-((x - (cx - offset))**2 + (y - cy)**2) / (2 * sigma_blob**2))
    # Right blob (pulling left)
    g_right = np.exp(-((x - (cx + offset))**2 + (y - cy)**2) / (2 * sigma_blob**2))

    # Traction magnitudes (typical cell traction: 100-1000 Pa)
    max_traction = 500.0  # Pa

    # tx: left blob pulls right (+), right blob pulls left (-)
    tx_gt = max_traction * g_left - max_traction * g_right
    # ty: add some vertical component (realistic cell shape)
    ty_gt = 0.3 * max_traction * g_left * (y - cy) / (sigma_blob * 2)
    ty_gt -= 0.3 * max_traction * g_right * (y - cy) / (sigma_blob * 2)

    return tx_gt.astype(np.float64), ty_gt.astype(np.float64)

def load_or_generate_data():
    """
    Generate synthetic benchmark data:
    1. Create ground truth traction field
    2. Apply forward model to get displacement field (in pixels)
    3. Add measurement noise
    """
    print("[DATA] Generating synthetic traction force dipole...")
    tx_gt, ty_gt = generate_synthetic_traction_field(N=GRID_SIZE)

    # Forward model: traction → displacement (returns pixels)
    print("[DATA] Applying forward operator (Boussinesq Green's function)...")
    u_clean_px, v_clean_px = forward_operator(tx_gt, ty_gt, PIXEL_SIZE, YOUNG_MODULUS, POISSON_RATIO)

    print(f"[DATA] Max displacement: {np.max(np.sqrt(u_clean_px**2 + v_clean_px**2)):.4f} pixels")

    # Add Gaussian noise to displacement
    np.random.seed(42)
    u_noisy_px = u_clean_px + NOISE_STD * np.random.randn(*u_clean_px.shape)
    v_noisy_px = v_clean_px + NOISE_STD * np.random.randn(*v_clean_px.shape)

    metadata = {
        "young": YOUNG_MODULUS,
        "sigma": POISSON_RATIO,
        "pixelsize1": PIXEL_SIZE,     # used by ffttc_traction to scale u,v from pixels to meters
        "pixelsize2": PIXEL_SIZE,     # used by ffttc_traction for wave vector computation
        "noise_std": NOISE_STD,
    }

    return (u_noisy_px, v_noisy_px), (tx_gt, ty_gt), metadata

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. Inverse Solver (Displacement → Traction)
# ═══════════════════════════════════════════════════════════
def reconstruct(measurements, **params):
    """
    FTTC inverse solver: recover traction forces from displacement field.

    Uses pyTFM's Fourier Transform Traction Cytometry with Gaussian filtering
    for regularization (noise suppression in high-frequency components).

    Parameters:
        measurements: tuple (u, v) displacement fields in pixels
        params: must include young, sigma, pixelsize1, pixelsize2

    Returns:
        tx_recon, ty_recon: reconstructed traction fields in Pa
    """
    u, v = measurements
    young = params["young"]
    sigma = params["sigma"]
    pixelsize1 = params["pixelsize1"]
    pixelsize2 = params["pixelsize2"]

    # Use pyTFM's FTTC solver
    # Note: spatial_filter=None for no filtering (best for low noise)
    # For noisy data, use spatial_filter="gaussian" with appropriate fs
    tx_recon, ty_recon = ffttc_traction(
        u, v,
        pixelsize1=pixelsize1,
        pixelsize2=pixelsize2,
        young=young,
        sigma=sigma,
        spatial_filter="gaussian",
        fs=3  # smaller filter for less smoothing (in auto-units)
    )

    return tx_recon, ty_recon

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `compute_ssim`, `compute_rmse`, `compute_relative_error`, `visualize_results`

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. Evaluation Metrics
# ═══════════════════════════════════════════════════════════
def compute_psnr(ref, test, data_range=None):
    """Compute PSNR (dB) between reference and test arrays."""
    if data_range is None:
        data_range = ref.max() - ref.min()
    if data_range == 0:
        return float('inf') if np.allclose(ref, test) else 0.0
    mse = np.mean((ref.astype(float) - test.astype(float)) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(data_range ** 2 / mse)

def compute_ssim(ref, test):
    """Compute SSIM for 2D fields."""
    from skimage.metrics import structural_similarity as ssim
    data_range = ref.max() - ref.min()
    if data_range == 0:
        data_range = 1.0
    return ssim(ref, test, data_range=data_range)

def compute_rmse(ref, test):
    """Compute RMSE."""
    return np.sqrt(np.mean((ref.astype(float) - test.astype(float)) ** 2))

def compute_relative_error(ref, test):
    """Compute relative error (RE) = ||ref - test||_2 / ||ref||_2."""
    ref_norm = np.linalg.norm(ref.ravel())
    if ref_norm == 0:
        return float('inf')
    return np.linalg.norm((ref - test).ravel()) / ref_norm

# ═══════════════════════════════════════════════════════════
# 6. Visualization
# ═══════════════════════════════════════════════════════════
def visualize_results(tx_gt, ty_gt, u_meas, v_meas, tx_rec, ty_rec, metrics, save_path):
    """
    Generate comprehensive visualization:
    Row 1: Ground truth traction magnitude | Measured displacement magnitude |
           Reconstructed traction magnitude | Error map
    Row 2: Quiver plots of GT tractions | measured displacements |
           reconstructed tractions | error vectors
    """
    gt_mag = np.sqrt(tx_gt**2 + ty_gt**2)
    rec_mag = np.sqrt(tx_rec**2 + ty_rec**2)
    disp_mag = np.sqrt(u_meas**2 + v_meas**2)
    err_mag = np.abs(gt_mag - rec_mag)

    vmax = max(gt_mag.max(), rec_mag.max())

    fig, axes = plt.subplots(2, 4, figsize=(24, 12))

    # Row 1: Scalar magnitude fields
    im0 = axes[0, 0].imshow(gt_mag, cmap='hot', vmin=0, vmax=vmax)
    axes[0, 0].set_title("Ground Truth |T| (Pa)", fontsize=12)
    plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

    im1 = axes[0, 1].imshow(disp_mag, cmap='viridis')
    axes[0, 1].set_title("Measured |u| (pixels)", fontsize=12)
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    im2 = axes[0, 2].imshow(rec_mag, cmap='hot', vmin=0, vmax=vmax)
    axes[0, 2].set_title("Reconstructed |T| (Pa)", fontsize=12)
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)

    im3 = axes[0, 3].imshow(err_mag, cmap='magma')
    axes[0, 3].set_title("Error |GT - Recon| (Pa)", fontsize=12)
    plt.colorbar(im3, ax=axes[0, 3], fraction=0.046)

    # Row 2: Quiver plots
    N = tx_gt.shape[0]
    skip = max(1, N // 16)  # subsample for readability
    y_grid, x_grid = np.mgrid[:N, :N]
    sl = (slice(None, None, skip), slice(None, None, skip))

    axes[1, 0].quiver(x_grid[sl], y_grid[sl], tx_gt[sl], ty_gt[sl],
                       gt_mag[sl], cmap='hot', scale_units='xy')
    axes[1, 0].set_title("GT Traction Vectors", fontsize=12)
    axes[1, 0].set_aspect('equal')
    axes[1, 0].invert_yaxis()

    axes[1, 1].quiver(x_grid[sl], y_grid[sl], u_meas[sl], v_meas[sl],
                       disp_mag[sl], cmap='viridis', scale_units='xy')
    axes[1, 1].set_title("Measured Displacement Vectors", fontsize=12)
    axes[1, 1].set_aspect('equal')
    axes[1, 1].invert_yaxis()

    axes[1, 2].quiver(x_grid[sl], y_grid[sl], tx_rec[sl], ty_rec[sl],
                       rec_mag[sl], cmap='hot', scale_units='xy')
    axes[1, 2].set_title("Reconstructed Traction Vectors", fontsize=12)
    axes[1, 2].set_aspect('equal')
    axes[1, 2].invert_yaxis()

    err_tx = tx_gt - tx_rec
    err_ty = ty_gt - ty_rec
    axes[1, 3].quiver(x_grid[sl], y_grid[sl], err_tx[sl], err_ty[sl],
                       err_mag[sl], cmap='magma', scale_units='xy')
    axes[1, 3].set_title("Error Vectors", fontsize=12)
    axes[1, 3].set_aspect('equal')
    axes[1, 3].invert_yaxis()

    fig.suptitle(
        f"pyTFM — Traction Force Microscopy Reconstruction\n"
        f"PSNR={metrics['psnr']:.2f} dB | SSIM={metrics['ssim']:.4f} | "
        f"RMSE={metrics['rmse']:.2f} Pa | RE={metrics['relative_error']:.4f} | "
        f"CC={metrics['correlation_coefficient']:.4f}",
        fontsize=14, fontweight='bold'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved visualization → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("  pyTFM — Traction Force Microscopy Inverse Problem")
print("=" * 60)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (a) Generate synthetic data
measurements, ground_truth, metadata = load_or_generate_data()
u_meas, v_meas = measurements
tx_gt, ty_gt = ground_truth
print(f"[DATA] Displacement field shape: {u_meas.shape}")
print(f"[DATA] GT traction field shape: {tx_gt.shape}")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# (b) Run inverse solver (FTTC)
print("[RECON] Running FTTC inverse solver...")
tx_rec, ty_rec = reconstruct(measurements, **metadata)
print(f"[RECON] Reconstructed traction field shape: {tx_rec.shape}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# (c) Compute metrics on traction magnitude
gt_mag = np.sqrt(tx_gt**2 + ty_gt**2)
rec_mag = np.sqrt(tx_rec**2 + ty_rec**2)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = {
    "psnr": float(compute_psnr(gt_mag, rec_mag)),
    "ssim": float(compute_ssim(gt_mag, rec_mag)),
    "rmse": float(compute_rmse(gt_mag, rec_mag)),
    "relative_error": float(compute_relative_error(gt_mag, rec_mag)),
    "correlation_coefficient": float(compute_correlation_coefficient(gt_mag, rec_mag)),
    # Also compute component-wise metrics
    "psnr_tx": float(compute_psnr(tx_gt, tx_rec)),
    "psnr_ty": float(compute_psnr(ty_gt, ty_rec)),
    "rmse_tx": float(compute_rmse(tx_gt, tx_rec)),
    "rmse_ty": float(compute_rmse(ty_gt, ty_rec)),
}

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"[EVAL] PSNR (magnitude) = {metrics['psnr']:.4f} dB")
print(f"[EVAL] SSIM (magnitude) = {metrics['ssim']:.6f}")
print(f"[EVAL] RMSE (magnitude) = {metrics['rmse']:.4f} Pa")
print(f"[EVAL] Relative Error   = {metrics['relative_error']:.6f}")
print(f"[EVAL] Correlation Coef = {metrics['correlation_coefficient']:.6f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# (d) Save metrics
metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"[SAVE] Metrics → {metrics_path}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (e) Visualize
vis_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
visualize_results(tx_gt, ty_gt, u_meas, v_meas, tx_rec, ty_rec, metrics, vis_path)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# (f) Save arrays
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), np.stack([tx_rec, ty_rec]))
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), np.stack([tx_gt, ty_gt]))
np.save(os.path.join(RESULTS_DIR, "measurements.npy"), np.stack([u_meas, v_meas]))

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("  DONE")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pyTFM**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=22.92 dB, SSIM=0.6963)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: empymod: Open-source full 3D electromagnetic modeller for 1D VTI media
- Repository: https://github.com/emsig/empymod
